<a href="https://colab.research.google.com/github/kubayishaun-droid/Baardman-Projects/blob/main/sequential_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Forecast California Housing Prices

In [3]:
# Import libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Dataset
from sklearn.datasets import fetch_california_housing

In [4]:
# Set random seed

np.random.seed(50)
torch.manual_seed(50)

In [5]:
!wget -O cal_housing.tgz "https://s3.amazonaws.com/sagemaker-sample-files/datasets/tabular/california_housing/cal_housing.tgz"

--2026-11-20 15:06:06--  https://s3.amazonaws.com/sagemaker-sample-files/datasets/tabular/california_housing/cal_housing.tgz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.217.122.128, 16.15.246.126, 54.231.227.72, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.217.122.128|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 441963 (432K) [application/x-tar]
Saving to: ‘cal_housing.tgz’

cal_housing.tgz     100%[===================>] 431.60K  1.44MB/s    in 0.3s    

2026-11-20 15:06:07 (1.44 MB/s) - ‘cal_housing.tgz’ saved [441963/441963]



In [6]:
!tar -xzf cal_housing.tgz
!ls

cal_housing.tgz  CaliforniaHousing  sample_data


In [7]:
!ls -lh CaliforniaHousing

total 2.0M
-rw-r--r-- 1 10017 166 2.0M Jul  7  2000 cal_housing.data
-rw-r--r-- 1 10017 166  237 Jul  7  2000 cal_housing.domain


In [8]:
import numpy as np
from sklearn.utils import Bunch

def fetch_california_housing():
  cal_housing = np.loadtxt(
  "CaliforniaHousing/cal_housing.data",
    delimiter=","
   )
  cal_housing = cal_housing[:, [8, 7, 2, 3, 4, 5, 6, 1, 0]]

  feature_names = [
      "MedInc", "HouseAge", "AveRooms", "AveBedrms",
      "Population", "AveOccup", "Latitude", "Longitude"
  ]

  target, housing_data = cal_housing[:, 0], cal_housing[:, 1:]

  housing_data[:, 2] /= housing_data[:, 5]
  housing_data[:, 3] /= housing_data[:, 5]
  housing_data[:, 5]  = housing_data[:, 4] / housing_data[:, 5]

  target = target / 100000.0

  return Bunch(
      data=housing_data,
      target=target,
      feature_names=feature_names
  )


In [9]:
# Load the dataset
data = fetch_california_housing()
features = pd.DataFrame(data.data, columns=data.feature_names)
targets = pd.Series(data.target, name='Price').values.astype(float).reshape(-1, 1)

In [10]:
print(features.shape)
print(targets.shape)
print(features.head)

(20640, 8)
(20640, 1)
<bound method NDFrame.head of        MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0      8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1      8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2      7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3      5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4      3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   
...       ...       ...       ...        ...         ...       ...       ...   
20635  1.5603      25.0  5.045455   1.133333       845.0  2.560606     39.48   
20636  2.5568      18.0  6.114035   1.315789       356.0  3.122807     39.49   
20637  1.7000      17.0  5.205543   1.120092      1007.0  2.325635     39.43   
20638  1.8672      18.0  5.329513   1.171920       741.0  2.123209     39.43   
20639  2.3886      16.0  5.254717   1.162264      1387.0  2.616981  

In [11]:
# Normalise features and target values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

features_normalized = scaler_X.fit_transform(features)
targets_normalized = scaler_y.fit_transform(targets.reshape(-1, 1))

**Questions**

* What is a `Bunch`?
* What is the default type for `data`?
* What is the default type for `target`?

**Answers**

1. A dictionery like object used by scikit-learn to store datasets and thier related information.
2. The default data type for data is Numpy array(numpy.ndarray)
3. The default data type for target is a Numpy array (numpy.ndarray).


In [12]:
def create_sequences(data, targets, seq_length):
   X, y = [], []
   for i in range(len(data) - seq_length):
       X.append(data[i:i+seq_length])
       y.append(targets[i+seq_length])
   return np.array(X), np.array(y)


seq_length = 20
X, y = create_sequences(features_normalized, targets_normalized, seq_length)

# Data cleaning and preprocessing

# LSTM model

In [13]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(LSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ =self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])

        return out

In [14]:
X_train, X_test, y_train, y_test =  train_test_split(
    X, y, test_size=0.33, random_state=50
)

In [15]:
# Converting data to Pytorch tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [16]:
# Defining the hyperparameters
input_size = X_train.shape[2]
hidden_size = 64
output_size = 1
num_layers = 2
learning_rate = 0.001
l2_lambda = 0.0001
num_epochs = 100

In [17]:
# Initilize the model
model = LSTMModel(input_size, hidden_size, output_size, num_layers)

# Define loss function
criterion = nn.MSELoss()

# Define optimiser with L2 regularisation
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=l2_lambda)

In [18]:
# Training the loop.
for epoch in range(num_epochs):
    model.train()

    optimizer.zero_grad()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    print(f'Epoch {epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch 1/100], Loss: 0.2115
Epoch 2/100], Loss: 0.1893
Epoch 3/100], Loss: 0.1689
Epoch 4/100], Loss: 0.1500
Epoch 5/100], Loss: 0.1322
Epoch 6/100], Loss: 0.1154
Epoch 7/100], Loss: 0.0997
Epoch 8/100], Loss: 0.0853
Epoch 9/100], Loss: 0.0728
Epoch 10/100], Loss: 0.0630
Epoch 11/100], Loss: 0.0574
Epoch 12/100], Loss: 0.0576
Epoch 13/100], Loss: 0.0636
Epoch 14/100], Loss: 0.0697
Epoch 15/100], Loss: 0.0711
Epoch 16/100], Loss: 0.0683
Epoch 17/100], Loss: 0.0641
Epoch 18/100], Loss: 0.0602
Epoch 19/100], Loss: 0.0577
Epoch 20/100], Loss: 0.0564
Epoch 21/100], Loss: 0.0562
Epoch 22/100], Loss: 0.0566
Epoch 23/100], Loss: 0.0572
Epoch 24/100], Loss: 0.0578
Epoch 25/100], Loss: 0.0584
Epoch 26/100], Loss: 0.0587
Epoch 27/100], Loss: 0.0587
Epoch 28/100], Loss: 0.0585
Epoch 29/100], Loss: 0.0582
Epoch 30/100], Loss: 0.0576
Epoch 31/100], Loss: 0.0570
Epoch 32/100], Loss: 0.0563
Epoch 33/100], Loss: 0.0557
Epoch 34/100], Loss: 0.0552
Epoch 35/100], Loss: 0.0548
Epoch 36/100], Loss: 0.0546
E

In [19]:
# Make predictions on the test data
model.eval()

with torch.no_grad():
    predictions = model(X_test)

# Convert predictions to Numpy arrays
predictions = predictions.numpy()
y_test_actual = y_test.numpy()

# Inverse tranformation the predictions and actual values
predictions = scaler_y.inverse_transform(predictions)
y_test_actual = scaler_y.inverse_transform(y_test_actual.reshape(-1, 1))

In [20]:
# Calculate Mean Squared Error

mse = mean_squared_error(y_test_actual, predictions)

print(f"Mean Squared Error (MSE): {mse:4f}")

Mean Squared Error (MSE): 0.657672


In [21]:
# Initialise LSTM without regularisation
model_lstm_no_reg = LSTMModel(input_size, hidden_size, output_size, num_layers)

criterion_lstm = nn.MSELoss()

optimizer_lstm = optim.Adam(
    model_lstm_no_reg.parameters(),
    lr=learning_rate
)

# Train the LSTM
for epoch in range(num_epochs):
    model_lstm_no_reg.train()

    optimizer_lstm.zero_grad()

    outputs = model_lstm_no_reg(X_train)
    loss = criterion_lstm(outputs, y_train)

    loss.backward()
    optimizer_lstm.step()

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/100], Loss: 0.2607
Epoch [2/100], Loss: 0.2409
Epoch [3/100], Loss: 0.2217
Epoch [4/100], Loss: 0.2026
Epoch [5/100], Loss: 0.1837
Epoch [6/100], Loss: 0.1647
Epoch [7/100], Loss: 0.1455
Epoch [8/100], Loss: 0.1261
Epoch [9/100], Loss: 0.1067
Epoch [10/100], Loss: 0.0879
Epoch [11/100], Loss: 0.0710
Epoch [12/100], Loss: 0.0587
Epoch [13/100], Loss: 0.0560
Epoch [14/100], Loss: 0.0668
Epoch [15/100], Loss: 0.0779
Epoch [16/100], Loss: 0.0776
Epoch [17/100], Loss: 0.0703
Epoch [18/100], Loss: 0.0625
Epoch [19/100], Loss: 0.0573
Epoch [20/100], Loss: 0.0552
Epoch [21/100], Loss: 0.0553
Epoch [22/100], Loss: 0.0565
Epoch [23/100], Loss: 0.0580
Epoch [24/100], Loss: 0.0593
Epoch [25/100], Loss: 0.0602
Epoch [26/100], Loss: 0.0605
Epoch [27/100], Loss: 0.0604
Epoch [28/100], Loss: 0.0597
Epoch [29/100], Loss: 0.0588
Epoch [30/100], Loss: 0.0576
Epoch [31/100], Loss: 0.0564
Epoch [32/100], Loss: 0.0553
Epoch [33/100], Loss: 0.0544
Epoch [34/100], Loss: 0.0540
Epoch [35/100], Loss: 0

In [22]:
# Evaluate LSTM without regularisation
model_lstm_no_reg.eval()

with torch.no_grad():
    predictions_lstm = model_lstm_no_reg(X_test)

predictions_lstm = scaler_y.inverse_transform(predictions_lstm.numpy())

mse_lstm_no_reg = mean_squared_error(y_test_actual, predictions_lstm)

print(f"LSTM without regulations MSE: {mse_lstm_no_reg:.6f}")

LSTM without regulations MSE: 0.668417


In [35]:
# LSTM Configuration A

hidden_size_A = 32
num_layers_A = 1
learning_rate_A = 0.001



In [37]:
# Initialise LSTM Configuration

model_lstm_A = LSTMModel(input_size, hidden_size_A, output_size, num_layers_A)

criterion_A = nn.MSELoss()

optimizer_A = optim.Adam(
    model_lstm_A.parameters(),
    lr=learning_rate_A,
    weight_decay=l2_lambda
)

In [40]:
# Train LSTM Configuration A

for epoch in range(num_epochs):
  model_lstm_A.train()
  optimizer_A.zero_grad()

  outputs = model_lstm_A(X_train)
  loss = criterion_A(outputs, y_train)

  loss.backward()
  optimizer_A.step()

  print(f'Epoch [{epoch + 1}/ {num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/ 100], Loss: 0.1731
Epoch [2/ 100], Loss: 0.1654
Epoch [3/ 100], Loss: 0.1580
Epoch [4/ 100], Loss: 0.1510
Epoch [5/ 100], Loss: 0.1444
Epoch [6/ 100], Loss: 0.1380
Epoch [7/ 100], Loss: 0.1320
Epoch [8/ 100], Loss: 0.1262
Epoch [9/ 100], Loss: 0.1206
Epoch [10/ 100], Loss: 0.1153
Epoch [11/ 100], Loss: 0.1102
Epoch [12/ 100], Loss: 0.1053
Epoch [13/ 100], Loss: 0.1006
Epoch [14/ 100], Loss: 0.0961
Epoch [15/ 100], Loss: 0.0917
Epoch [16/ 100], Loss: 0.0876
Epoch [17/ 100], Loss: 0.0836
Epoch [18/ 100], Loss: 0.0798
Epoch [19/ 100], Loss: 0.0763
Epoch [20/ 100], Loss: 0.0729
Epoch [21/ 100], Loss: 0.0698
Epoch [22/ 100], Loss: 0.0670
Epoch [23/ 100], Loss: 0.0644
Epoch [24/ 100], Loss: 0.0622
Epoch [25/ 100], Loss: 0.0603
Epoch [26/ 100], Loss: 0.0589
Epoch [27/ 100], Loss: 0.0578
Epoch [28/ 100], Loss: 0.0573
Epoch [29/ 100], Loss: 0.0571
Epoch [30/ 100], Loss: 0.0574
Epoch [31/ 100], Loss: 0.0579
Epoch [32/ 100], Loss: 0.0585
Epoch [33/ 100], Loss: 0.0590
Epoch [34/ 100], Lo

In [41]:
# Evaluate LSTM Configuration A

model_lstm_A.eval()

with torch.no_grad():
  predictions_A = model_lstm_A(X_test)

  predictions_A = predictions_A.numpy()

  predictions_A = scaler_y.inverse_transform(predictions_A)

  mse_lstm_A = mean_squared_error(y_test_actual, predictions_A)

  print(f"LSTM Configuration A MSE: {mse_lstm_A:.6f}")


LSTM Configuration A MSE: 1.151278


In [24]:
import torch
print(torch.cuda.is_available())

True


In [25]:
import torch
print(torch.cuda.is_available())

True


In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [27]:
# LSTM Configuration B

hidden_size_B = 128
num_layers_B = 2
learning_rate_B = 0.0005

model_lstm_B = LSTMModel(input_size, hidden_size_B, output_size, num_layers_B)

criterion_B = nn.MSELoss()

optimizer_B = optim.Adam(
    model_lstm_B.parameters(),
    lr=learning_rate_B,
    weight_decay=l2_lambda
)

for epoch in range(num_epochs):
    model_lstm_B.train()
    optimizer_B.zero_grad()

    outputs = model_lstm_B(X_train)
    loss = criterion_B(outputs, y_train)

    loss.backward()
    optimizer_B.step()

model_lstm_B.eval()

with torch.no_grad():
    predictions_B = model_lstm_B(X_test).numpy()

predictions_B = scaler_y.inverse_transform(predictions_B)

mse_lstm_B = mean_squared_error(y_test_actual, predictions_B)

print(f"LSTM Configuration B MSE: {mse_lstm_B:.6f}")

LSTM Configuration B MSE: 0.698419


# GRU model

In [28]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(GRUModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _= self.gru(x, h0)
        out = self.fc(out[:, -1, :])

        return out

In [29]:
# Initialise GRU without regularisation

model_gru = GRUModel(input_size, hidden_size, output_size, num_layers)
criterion_gru = nn.MSELoss()

optimizer_gru = optim.Adam(
    model_gru.parameters(),
    lr=learning_rate
)

#Train the GRU
for epoch in range(num_epochs):
    model_gru.train()

    optimizer_gru.zero_grad()

outputs = model_gru(X_train)
loss = criterion_gru(outputs, y_train)

loss.backward()
optimizer_gru.step()

print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [100/100], Loss: 0.1046


In [30]:
# Evaluate GRU without regularisation

model_gru.eval()

with torch.no_grad():
    predictions_gru = model_gru(X_test)

# Comvert predictions to Numpy arrays
predictions_gru = predictions_gru.numpy()

# Inverse tranform predictions
predictions_gru = scaler_y.inverse_transform(predictions_gru)

# Calculate MSE
mse_gru_no_reg = mean_squared_error(y_test_actual, predictions_gru)

print(f"GRU without regulation MSE: {mse_gru_no_reg:.6f}")


GRU without regulation MSE: 1.825416


In [31]:
# Initialise GRU with regularistion.
model_gru_reg = GRUModel(input_size, hidden_size, output_size, num_layers)

optimizer_gru_reg = optim.Adam(
    model_gru_reg.parameters(),
    lr=learning_rate,
    weight_decay=l2_lambda
)

# Train the GRU
for epoch in range(num_epochs):
    model_gru_reg.train()
    optimizer_gru_reg.zero_grad()

    outputs = model_gru_reg(X_train)
    loss = criterion_gru(outputs, y_train)

    loss.backward()
    optimizer_gru_reg.step()


In [32]:
# Evaluate GRU regularisation
model_gru_reg.eval()

with torch.no_grad():
    predictions_gru_reg = model_gru_reg(X_test)

predictions_gru_reg = predictions_gru_reg.numpy()

predictions_gru_reg = scaler_y.inverse_transform(predictions_gru_reg)

mse_gru_reg = mean_squared_error(y_test_actual, predictions_gru_reg)

print(f"GRU with regulation MSE: {mse_gru_reg:.6}")

GRU with regulation MSE: 0.625767


In [44]:
# GRU Configuration A

hidden_size_A = 32
num_layers_A = 1
learning_rate_A = 0.001

model_gru_A = GRUModel(
    input_size,
    hidden_size_A,
    output_size,
    num_layers_A
)

criterion_gru_A = nn.MSELoss()

optimizer_gru_A = optim.Adam(
    model_gru_A.parameters(),
    lr=learning_rate_A,
    weight_decay=l2_lambda
)


In [45]:
# Train GRU Configuration A

for epoch in range (num_epochs):
  model_gru_A.train()
  optimizer_gru_A.zero_grad()

  outputs = model_gru_A(X_train)
  loss = criterion_gru_A(outputs, y_train)

  loss.backward()
  optimizer_gru_A.step()

In [46]:
# Evaluate GRU Configuration A

model_gru_A.eval()

with torch.no_grad():
  predictions_gru_A = model_gru_A(X_test)

  predictions_gru_A = scaler_y.inverse_transform(predictions_gru_A)

  mse_gru_A = mean_squared_error(y_test_actual, predictions_gru_A)

  print(f"GRU Configuration A MSE: {mse_gru_A:.6f}")

GRU Configuration A MSE: 0.770968


In [48]:
# GRU Configuration B

hidden_size_B = 128
num_layers_B = 2
learning_rate_B = 0.0005

model_gru_B = GRUModel(
    input_size,
    hidden_size_B,
    output_size,
    num_layers_B
)

criterion_gru_B = nn.MSELoss()

optimizer_gru_B = optim.Adam(
    model_gru_B.parameters(),
    lr=learning_rate_B,
    weight_decay=l2_lambda
)

In [51]:
# Train GRU Configuration B

for epoch in range(num_epochs):
  model_gru_B.train()
  optimizer_gru_B.zero_grad()

  outputs = model_gru_B(X_train)
  loss = criterion_gru_B(outputs, y_train)

  loss.backward()
  optimizer_gru_B.step()

In [52]:
# Evaluate GRU Configuration B

model_gru_B.eval()

with torch.no_grad():
  predictions_gru_B = model_gru_B(X_test)

  predictions_gru_B =predictions_gru_B.numpy()

  predictions_gru_B = scaler_y.inverse_transform(predictions_gru_B)

  mse_gru_B = mean_squared_error(y_test_actual, predictions_gru_B)

  print(f"GRU Configuration B MSE: {mse_gru_B:.6f}")

GRU Configuration B MSE: 0.635258


A. No, The LSTM performed slightly better without regularisation, with its MSE decreasing from 0.656627 to 0.640275. However, the GRU also improved significantly with regularisation, with its MSE decreasing from 1.825416 to 0.625767.

B. Regularisation helps prevent overfitting by controlling the complexity of a model. It improves the model's ability to generalise to unseen data and can reduce prediction errors. The GRU results demonstrate the importance of regularisation in this experiment.

### Question 3: Hyperparmeter Optimisation

### LSTM Model:**
- Configuration A: 32 hidden units, 1 layer, learning rate of 0.001. MSE: 1.151278.
-Configuration B: 128 hidden units, 2 layers, learning rate of 0.0005. MSE: 0.698419.

Configuration B performed better, reducing the predcition error.

**GRU Model:**
-Configuration A: MSE of 0.770968.
-Configuration B: MSE of 0.633258.

Configuration B also performed better for the GRU model.

**Conclusion:**

Both models improved after adjusting thier hyperparmeters. The GRU Configuration B achieved the lowest MSE of 0.633258, making it the best-performing configuration in this experiment.

**Note:** The California housing dataset was sourced by scikit-learn from the StatLib repository: https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

**Reference**  
Pace, R. K., & Barry, R. (1997). Sparse spatial autoregressions. Statistics & Probability Letters, 33(3), 291-297.